In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "/workspaces/dev/modules/python-utils",
    "/workspaces/dev/modules/ai-utils",
    "/workspaces/dev/test/modules/whisper_streaming",
    "/workspaces/dev/test/performance_test/libri",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path
import numpy as np

In [ ]:
from sj_ai_utils.datasets.libri_speech_asr_corpus.service import search_dirs
from sj_ai_utils.datasets.libri_speech_asr_corpus.sclite import generate_trn
from sj_ai_utils.evaluator.sclite_utils import sclite_trn, parse_sclite_summary
from sj_utils.evaluator import TimeChecker

In [ ]:
from whisper_online import FasterWhisperASR, OnlineASRProcessor
from util import get_whisper_streaming_transcriber, normalize_text

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/test/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000
RANDOM_STATE = 42

In [ ]:
src = Path(SOURCE)

In [ ]:
data_paths = search_dirs(src)

In [ ]:
transcribe_time = TimeChecker()
processed_time = TimeChecker()

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
asr = FasterWhisperASR("en", MODEL_SIZE)
asr.use_vad()
online = OnlineASRProcessor(asr)
_transcriber = get_whisper_streaming_transcriber(online, SAMPLE_RATE, rng)
transcriber = lambda audio: _transcriber(audio, transcribe_time)

In [ ]:
processed_time.start()
data = generate_trn(data_paths, transcriber, normalize_text, 1)
processed_time.check()

In [ ]:
concat_result = {}
for value in data.values():
    for k, v in value.items():
        if k not in concat_result:
            concat_result[k] = []
        concat_result[k].extend(v)

In [ ]:
output = sclite_trn(
    concat_result["ref"],
    concat_result["hyp"],
)

In [ ]:
{
    "result":parse_sclite_summary(output),
    "processed_time": processed_time.metric(),
    "transcribe_time": transcribe_time.metric(),
}